In [0]:
from pyspark.sql import functions as F

# ==========================================
# CONFIGURACIÓN
# ==========================================

TABLA_EVALUADA = "quality_data.gold.control_calidad_evaluado"

TABLA_RESUMEN_ALERTAS = "quality_data.gold.resumen_alertas_calidad"
TABLA_ALERTAS_DETALLE = "quality_data.gold.alertas_calidad"
TABLA_KPI_GENERAL = "quality_data.gold.kpi_control_calidad"

df_evaluado = spark.table(TABLA_EVALUADA)

print("=" * 60)
print("CARGA DE TABLA EVALUADA")
print("=" * 60)
print(f"Tabla origen: {TABLA_EVALUADA}")
print(f"Registros: {df_evaluado.count():,}")

display(df_evaluado.limit(20))

In [0]:
# ==========================================
# DETALLE DE ALERTAS FUERA DE LÍMITE
# ==========================================

df_alertas = (
    df_evaluado
    .filter(F.col("es_fuera_limite") == True)
    .select(
        "hdr",
        "cliente_codigo",
        "articulo_codigo",
        "variable_nombre",
        "resultado_original",
        "resultado_numerico",
        "limite_inferior",
        "limite_superior",
        "estado_control",
        "distancia_al_limite",
        "color_codigo",
        "color_nombre",
        "operacion_codigo",
        "fecha_medicion",
        "fecha_medicion_dia",
        "anio_medicion",
        "mes_medicion",
        "semana_medicion",
        "total_mediciones",
        "promedio",
        "desviacion_estandar",
        "metodo_calculo",
        "fecha_evaluacion"
    )
)

spark.sql("CREATE SCHEMA IF NOT EXISTS quality_data.gold")
spark.sql(f"DROP TABLE IF EXISTS {TABLA_ALERTAS_DETALLE}")

df_alertas.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLA_ALERTAS_DETALLE)

print(f"✅ Tabla creada: {TABLA_ALERTAS_DETALLE}")
print(f"Alertas detectadas: {df_alertas.count():,}")

display(df_alertas.limit(100))

In [0]:
# ==========================================
# RESUMEN DE ALERTAS PARA DASHBOARD
# ==========================================

df_resumen_alertas = (
    df_evaluado
    .groupBy(
        "articulo_codigo",
        "variable_nombre"
    )
    .agg(
        F.count("*").alias("total_mediciones"),
        F.sum(
            F.when(F.col("es_fuera_limite") == True, 1).otherwise(0)
        ).alias("total_fuera_limite"),
        F.round(
            F.sum(
                F.when(F.col("es_fuera_limite") == True, 1).otherwise(0)
            ) / F.count("*") * 100,
            2
        ).alias("porcentaje_fuera_limite"),
        F.round(F.avg("resultado_numerico"), 6).alias("promedio_resultado"),
        F.min("resultado_numerico").alias("valor_minimo"),
        F.max("resultado_numerico").alias("valor_maximo"),
        F.min("limite_inferior").alias("limite_inferior"),
        F.max("limite_superior").alias("limite_superior"),
        F.min("fecha_medicion").alias("fecha_minima"),
        F.max("fecha_medicion").alias("fecha_maxima")
    )
    .orderBy(F.desc("total_fuera_limite"))
)

spark.sql(f"DROP TABLE IF EXISTS {TABLA_RESUMEN_ALERTAS}")

df_resumen_alertas.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLA_RESUMEN_ALERTAS)

print(f"✅ Tabla creada: {TABLA_RESUMEN_ALERTAS}")

display(df_resumen_alertas)

In [0]:
# ==========================================
# KPIs GENERALES DEL DASHBOARD
# ==========================================

df_kpi_general = (
    df_evaluado
    .agg(
        F.count("*").alias("total_mediciones"),
        F.sum(
            F.when(F.col("es_fuera_limite") == True, 1).otherwise(0)
        ).alias("total_fuera_limite"),
        F.sum(
            F.when(F.col("estado_control") == "BAJO_LIMITE", 1).otherwise(0)
        ).alias("total_bajo_limite"),
        F.sum(
            F.when(F.col("estado_control") == "SUPERA_LIMITE", 1).otherwise(0)
        ).alias("total_sobre_limite"),
        F.countDistinct("articulo_codigo").alias("total_articulos"),
        F.countDistinct("variable_nombre").alias("total_variables"),
        F.min("fecha_medicion").alias("fecha_minima"),
        F.max("fecha_medicion").alias("fecha_maxima")
    )
    .withColumn(
        "porcentaje_fuera_limite",
        F.round(
            F.col("total_fuera_limite") / F.col("total_mediciones") * 100,
            2
        )
    )
    .withColumn(
        "fecha_actualizacion",
        F.current_timestamp()
    )
)

spark.sql(f"DROP TABLE IF EXISTS {TABLA_KPI_GENERAL}")

df_kpi_general.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLA_KPI_GENERAL)

print(f"✅ Tabla creada: {TABLA_KPI_GENERAL}")

display(df_kpi_general)

In [0]:
# ==========================================
# VALIDACIÓN FINAL
# ==========================================

tablas_dashboard = [
    TABLA_ALERTAS_DETALLE,
    TABLA_RESUMEN_ALERTAS,
    TABLA_KPI_GENERAL
]

for tabla in tablas_dashboard:
    df_temp = spark.table(tabla)
    print(f"{tabla}: {df_temp.count():,} registros")